# SE 系统效率因子 — 数据分析
Hydraulic Systems Dataset | 1 Hz 采样 | 2205 周期 × 60 秒/周期 | 虚拟传感器 (%)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = r"C:\Users\Lenovo\Desktop\项目集合\Hydraulic Systems dataset"

In [ ]:
# 加载 SE 和标签文件
se = pd.read_csv(os.path.join(DATA_DIR, "SE.txt"), sep="\t", header=None)
profile = pd.read_csv(os.path.join(DATA_DIR, "profile.txt"), sep="\t", header=None)
profile.columns = ["Cooler", "Valve", "Pump_Leakage", "Accumulator", "Stable"]

print(f"SE 形状: {se.shape} — {se.shape[0]} 个周期, {se.shape[1]} 个时间点/周期")
print(f"Profile 形状: {profile.shape}")

# 检查 SE 数据结构: 每周期前几列的值分布
print(f"\n每周期第0列 (初始值):  min={se[0].min():.1f}  max={se[0].max():.1f}  mean={se[0].mean():.1f}")
print(f"每周期第1-8列 (零值区): 非零比例={(se.iloc[:, 1:9].values != 0).mean()*100:.1f}%")
print(f"每周期第9-59列 (工作段): min={se.iloc[:, 9:60].values.min():.1f}  max={se.iloc[:, 9:60].values.max():.1f}")
print(f"\n标签值域:\n{profile.describe().round(1)}")

**数据特征**: 每周期 col 0 为初始效率值，col 1-8 为系统空转段（全 0），col 9-59 为有效工作段（连续重复值，有效采样约 0.5 Hz）。SE 作为系统级效率指标，理论上对所有导致能量损失的故障都有响应。

## 1. 效率曲线形态

In [ ]:
# 随机 6 个周期的 SE 效率曲线
np.random.seed(42)
sample_ids = np.random.choice(se.shape[0], 6, replace=False)
time = np.arange(0, 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, idx in zip(axes.flat, sample_ids):
    ax.plot(time, se.iloc[idx, :], linewidth=1.2, color="darkorange")
    # 标注零值区
    ax.axvspan(1, 9, alpha=0.15, color='gray')
    ax.axvline(x=9, color='gray', linestyle='--', alpha=0.6, linewidth=1)
    ax.set_title(f"周期 {idx+1}  —  Cooler={profile.loc[idx,'Cooler']:.0f}%  Pump={profile.loc[idx,'Pump_Leakage']:.0f}")
    ax.set_xlabel("时间 (秒)")
    ax.set_ylabel("效率 (%)")
    ax.grid(True, alpha=0.3)
plt.suptitle("随机 6 个周期的 SE 效率曲线 (灰色=空转段, 虚线=工作段起点)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 所有周期叠加，对比工作段 vs 空转段
time = np.arange(0, 60)
mean_all = se.mean(axis=0)
std_all = se.std(axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time, mean_all, linewidth=1.5, color="darkorange")
ax.fill_between(time, mean_all - std_all, mean_all + std_all, alpha=0.15, color="darkorange")
ax.axvspan(1, 9, alpha=0.15, color='gray')
ax.axvline(x=9, color='gray', linestyle='--', alpha=0.6, linewidth=1)
ax.set_title("所有 2205 周期 SE 均值曲线 (±1σ 阴影)", fontsize=13)
ax.set_xlabel("时间 (秒)")
ax.set_ylabel("效率 (%)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**形态特征**: col 0 有初始读数 → col 1-8 系统空转归零 → col 9 起进入有效工作段，效率在 ~55%-95% 间波动。曲线呈阶梯状（连续重复值），与 VS1 的连续振动波形完全不同。

## 2. 故障 × 效率损失量化

In [ ]:
# 定义工作段: col 9-59
work_se = se.iloc[:, 9:60]

# 计算每周期效率指标
se_metrics = pd.DataFrame({
    "init_eff": se[0],                          # 初始效率
    "work_mean": work_se.mean(axis=1),           # 工作段均值
    "work_min": work_se.min(axis=1),             # 工作段最低效率
    "work_max": work_se.max(axis=1),             # 工作段最高效率
    "work_std": work_se.std(axis=1),             # 工作段波动
    "loss_rate": 100 - work_se.mean(axis=1),     # 效率损失率 (%)
})

# 合并标签
for col in profile.columns:
    se_metrics[col] = profile[col].values

print("全量统计:")
print(f"  工作段均值: {se_metrics['work_mean'].mean():.1f}% ± {se_metrics['work_mean'].std():.1f}%")
print(f"  效率损失率: {se_metrics['loss_rate'].mean():.1f}% ± {se_metrics['loss_rate'].std():.1f}%")
se_metrics[['init_eff', 'work_mean', 'loss_rate']].describe().round(1)

In [ ]:
# 按四种故障分组，效率损失率 bar chart
fault_config = {
    "Cooler":       {3: "接近失效", 20: "效率降低", 100: "正常"},
    "Valve":        {73: "接近失效", 80: "严重滞后", 90: "轻微滞后", 100: "正常"},
    "Pump_Leakage": {0: "无泄漏", 1: "轻微泄漏", 2: "严重泄漏"},
    "Accumulator":  {90: "接近失效", 100: "严重降压", 115: "轻微降压", 130: "正常"},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#d73027', '#fc8d59', '#fee090', '#4575b4'][::-1]

for ax, (col, labels) in zip(axes.flat, fault_config.items()):
    means = []
    errs = []
    names = []
    for val, name in labels.items():
        mask = se_metrics[col] == val
        if mask.sum() > 0:
            means.append(se_metrics.loc[mask, 'loss_rate'].mean())
            errs.append(se_metrics.loc[mask, 'loss_rate'].std())
            names.append(f"{name}\n({val})")
    # 按 loss 降序排列
    order = np.argsort(means)[::-1]
    means = [means[i] for i in order]
    errs = [errs[i] for i in order]
    names = [names[i] for i in order]
    
    bars = ax.bar(names, means, color=colors[:len(names)], edgecolor='white')
    ax.errorbar(range(len(names)), means, yerr=errs, fmt='none', ecolor='black', capsize=5, alpha=0.5)
    ax.set_title(f"{col} — 各状态效率损失率")
    ax.set_ylabel("效率损失率 (%)")
    ax.grid(True, alpha=0.2, axis='y')
    # 在柱上标注数值
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{m:.1f}%', 
                ha='center', fontsize=9, fontweight='bold')

plt.suptitle("四种故障状态下的系统效率损失率 (误差线=±1σ)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 效率损失变化幅度 — 故障最严重 vs 正常
worst_best = {
    "Cooler":       (3, 100),
    "Valve":        (73, 100),
    "Pump_Leakage": (2, 0),      # 2=严重泄漏, 0=无泄漏
    "Accumulator":  (90, 130),
}

print("效率损失变化 (最严重故障 vs 正常):")
change_data = []
for col, labels in fault_config.items():
    worst_val, best_val = worst_best[col]
    worst_loss = se_metrics[se_metrics[col] == worst_val]['loss_rate'].mean()
    best_loss  = se_metrics[se_metrics[col] == best_val]['loss_rate'].mean()
    delta = worst_loss - best_loss
    print(f"  {col:<14}: {labels[worst_val]:>6} → {labels[best_val]:>4}  |  Δ损失率 = {delta:+.1f}%")
    change_data.append({'故障': col, 'Δ损失率': delta})

print(f"\n结论: 效率损失对哪些故障最敏感?")
for d in sorted(change_data, key=lambda x: x['Δ损失率'], reverse=True):
    print(f"  {d['故障']}: Δ={d['Δ损失率']:+.1f}%")

## 3. 周期级趋势与退化检测

In [ ]:
# 周期级效率均值 + 50 周期滚动平滑
se_metrics['cycle_idx'] = np.arange(len(se_metrics))
se_metrics['rolling_mean'] = se_metrics['work_mean'].rolling(window=50, center=True).mean()

fault_items = [
    ("Cooler",       {100: "正常 100%", 20: "效率降低 20%", 3: "接近失效 3%"}),
    ("Valve",        {100: "正常", 90: "轻微滞后", 80: "严重滞后", 73: "接近失效"}),
    ("Pump_Leakage", {0: "无泄漏", 1: "轻微泄漏", 2: "严重泄漏"}),
    ("Accumulator",  {130: "正常 130bar", 115: "轻微降压 115", 100: "严重降压 100", 90: "接近失效 90"}),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (col, labels) in zip(axes.flat, fault_items):
    for val, name in labels.items():
        mask = se_metrics[col] == val
        if mask.sum() > 0:
            sub = se_metrics[mask]
            ax.scatter(sub['cycle_idx'], sub['work_mean'], s=3, alpha=0.4, label=name)
    ax.plot(se_metrics['cycle_idx'], se_metrics['rolling_mean'], color='black', linewidth=1.5, alpha=0.6)
    ax.set_xlabel("周期编号")
    ax.set_ylabel("工作段效率均值 (%)")
    ax.set_title(f"{col} — 周期级效率变化")
    ax.legend(fontsize=8, markerscale=3)
    ax.grid(True, alpha=0.3)
plt.suptitle("SE 效率均值随周期数演变 — 按故障状态着色 (黑线=50周期滚动均值)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cooler 状态切换点前后 ±30 周期的效率变化
cooler_vals = profile["Cooler"].values
switch_points = np.where(np.diff(cooler_vals) != 0)[0]
print(f"Cooler 状态切换次数: {len(switch_points)} 次")

n_show = min(len(switch_points), 8)
if n_show > 0:
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes_arr = [axes] if n_show == 1 else axes.flat
    
    for i in range(n_show):
        sp = switch_points[i]
        start = max(0, sp - 30)
        end = min(len(se), sp + 30)
        local = se_metrics.iloc[start:end]
        for val, name in [(3, "失效"), (20, "降低"), (100, "正常")]:
            mask_local = local["Cooler"] == val
            if mask_local.sum() > 0:
                sub = local[mask_local]
                axes_arr[i].scatter(sub['cycle_idx'], sub['work_mean'], s=8, alpha=0.7, label=name)
        axes_arr[i].axvline(x=sp, color='black', linestyle='--', linewidth=1.5)
        axes_arr[i].set_title(f"切换点 #{i+1} (周期 {sp})", fontsize=9)
        axes_arr[i].legend(fontsize=6, markerscale=2)
        axes_arr[i].grid(True, alpha=0.3)
    
    for j in range(n_show, rows * cols):
        if hasattr(axes, 'flat'):
            axes.flat[j].set_visible(False)
    plt.suptitle("Cooler 状态切换前后 ±30 周期的效率变化", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("未检测到 Cooler 状态切换点")

## 4. 故障分类建模 — SE 能诊断哪些故障？

In [ ]:
# SE 特征提取
def extract_se_features(row):
    """
    提取 SE 传感器特有特征:
    - 初始效率 (col 0)
    - 工作段统计 (col 9-59)
    - 效率下降幅度
    - 前后半段效率差异
    """
    init = row[0]
    work = row[9:60].values
    
    # 将工作段分为前半/后半 (各 25/26 个点)
    half = len(work) // 2
    work_first = work[:half]
    work_second = work[half:]
    
    features = {
        'init_eff':        init,                              # 初始效率
        'work_mean':       work.mean(),                       # 工作段均值
        'work_min':        work.min(),                        # 工作段最低值
        'work_max':        work.max(),                        # 工作段最高值
        'work_std':        work.std(),                        # 工作段波动
        'work_median':     np.median(work),                   # 工作段中位数
        'drop_init2mean':  init - work.mean(),                # 初始→工作段效率降幅
        'first_half_mean': work_first.mean(),                 # 前半段均值
        'second_half_mean': work_second.mean(),               # 后半段均值
        'eff_recovery':    work_second.mean() - work_first.mean(),  # 效率恢复 (后-前)
        'work_range':      work.max() - work.min(),           # 工作段极差
        'skew':            sp_stats.skew(work),               # 工作段偏度
        'kurt':            sp_stats.kurtosis(work),           # 工作段峰度
    }
    return pd.Series(features)

X = pd.DataFrame([extract_se_features(se.iloc[i]) for i in range(len(se))])
print(f"特征矩阵: {X.shape[0]} 周期 × {X.shape[1]} 特征")
print(f"特征列表: {list(X.columns)}")
X.head()

In [ ]:
# 对四种故障分别做 RF 多分类
target_config = {
    "Cooler":       [3, 20, 100],
    "Valve":        [73, 80, 90, 100],
    "Pump_Leakage": [0, 1, 2],
    "Accumulator":  [90, 100, 115, 130],
}

results = {}
for name, classes in target_config.items():
    valid_mask = profile[name].isin(classes)
    X_sub = X[valid_mask]
    y_raw = profile[name].values[valid_mask]

    le = LabelEncoder()
    y_enc = le.fit_transform(y_raw)

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(rf, X_sub, y_enc, cv=5, scoring='f1_weighted')

    rf.fit(X_sub, y_enc)
    importances = rf.feature_importances_
    top_idx = np.argsort(importances)[-5:][::-1]

    results[name] = {
        'f1_mean': scores.mean(),
        'f1_std':  scores.std(),
        'top_features': list(zip(X.columns[top_idx], importances[top_idx]))
    }
    print(f"{name:<14} → 5折CV F1 = {scores.mean():.4f} ± {scores.std():.4f}")

print("\nSE 效率因子对各故障的分类能力:")
for name in ["Cooler", "Valve", "Pump_Leakage", "Accumulator"]:
    bar = "█" * int(results[name]['f1_mean'] * 40)
    print(f"  {name:<14}: {results[name]['f1_mean']:.3f}  {bar}")

print("\n各目标 Top-3 重要特征:")
for name, r in results.items():
    top3 = r['top_features'][:3]
    print(f"  {name}: {', '.join(f'{f} ({imp:.3f})' for f, imp in top3)}")

In [ ]:
# 最佳分类目标的特征重要性和混淆矩阵
best_name = max(results, key=lambda k: results[k]['f1_mean'])
valid_mask = profile[best_name].isin(target_config[best_name])
X_best = X[valid_mask]
y_best = profile[best_name].values[valid_mask]
le = LabelEncoder()
y_enc = le.fit_transform(y_best)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_best, y_enc)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5.5))

# 特征重要性
colors_imp = ['steelblue' if 'half' not in f else 'coral' for f in imp.index]
ax1.barh(imp.index[::-1], imp.values[::-1], color=[c for c in colors_imp[::-1]])
ax1.set_title(f"SE 特征重要性 — {best_name} 分类", fontsize=12)
ax1.set_xlabel("Importance")
ax1.grid(True, alpha=0.3, axis='x')

# 混淆矩阵
X_train, X_test, y_train, y_test = train_test_split(X_best, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_train, y_train)
y_pred = rf2.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

im = ax2.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax2.text(j, i, cm[i, j], ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if cm[i, j] > cm.max()/2 else "gray")
ax2.set_xticks(range(len(le.classes_)))
ax2.set_xticklabels(le.classes_)
ax2.set_yticks(range(len(le.classes_)))
ax2.set_yticklabels(le.classes_)
ax2.set_xlabel("预测")
ax2.set_ylabel("真实")
ax2.set_title(f"{best_name} 混淆矩阵", fontsize=12)
plt.colorbar(im, shrink=0.8, ax=ax2)
plt.suptitle(f"SE 效率因子 — {best_name} 故障诊断详细结果", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

In [ ]:
# 总结: SE vs 各故障的 F1 水平
f1_df = pd.DataFrame([
    {'故障': n, 'F1': r['f1_mean'], 'F1_std': r['f1_std']}
    for n, r in results.items()
]).sort_values('F1', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(f1_df['故障'], f1_df['F1'], color=['steelblue', 'coral', 'seagreen', 'darkorange'], edgecolor='white')
ax.errorbar(range(len(f1_df)), f1_df['F1'], yerr=f1_df['F1_std'], fmt='none', ecolor='black', capsize=5)
for bar, f1 in zip(bars, f1_df['F1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{f1:.3f}', 
            ha='center', fontsize=11, fontweight='bold')
ax.set_title("SE 效率因子 — 四种故障分类 F1 对比", fontsize=13)
ax.set_ylabel("F1 (weighted, 5-fold CV)")
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()